# P&ID Parsing and Graph Construction

Reads a DXF P&ID file, extracts text/symbol/pipe entities, cleans coordinates, clusters nodes and generates a NetworkX process graph.


In [ ]:
import numpy as np

def find_tag_xy(df_text, tag):
    pat = rf"\b{tag.replace('-','[-\\s]?')}\b"
    hits = df_text[df_text["text"].astype(str).str.contains(pat, case=False, regex=True)]
    if len(hits) == 0:
        raise ValueError(f"Tag not found: {tag}")
    return hits.iloc[0][["x","y"]].values.astype(float)

def nearest_insert_index(df_main, xy):
    pts = df_main[["x","y"]].values.astype(float)
    d = np.sqrt(((pts - xy)**2).sum(axis=1))
    return df_main.iloc[np.argmin(d)].name

reactor_tags = ["R-100","R-200","R-300"]
reactor_nodes = {}

for t in reactor_tags:
    xy = find_tag_xy(df_text, t)
    ins_idx = nearest_insert_index(df_main, xy)
    reactor_nodes[t] = f"ins_{ins_idx}"
    # stamp role
    G.nodes[reactor_nodes[t]]["unit_tag"] = t
    G.nodes[reactor_nodes[t]]["unit_type"] = "reactor"

G.nodes[reactor_nodes["R-300"]]["reactor_stage"] = "catalytic_fixed_bed"
G.nodes[reactor_nodes["R-100"]]["reactor_stage"] = "acidification"
G.nodes[reactor_nodes["R-200"]]["reactor_stage"] = "acidification"

reactor_nodes

In [ ]:
!pip -q install ezdxf shapely networkx pandas matplotlib
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install torch-geometric -q


In [ ]:
from google.colab import files
uploaded = files.upload()   # upload your .dxf
DXF_FILE = [k for k in uploaded.keys() if k.lower().endswith(".dxf")][0]
DXF_FILE


In [ ]:
import ezdxf
import pandas as pd
import numpy as np

doc = ezdxf.readfile(DXF_FILE)
msp = doc.modelspace()

# 1A) TEXT / MTEXT
texts = []
for e in msp.query("TEXT MTEXT"):
    if e.dxftype() == "TEXT":
        s = e.dxf.text
        x, y = e.dxf.insert.x, e.dxf.insert.y
        layer = e.dxf.layer
    else:
        s = e.text
        x, y = e.dxf.insert.x, e.dxf.insert.y
        layer = e.dxf.layer
    texts.append({"text": str(s).strip(), "x": x, "y": y, "layer": layer})

df_text = pd.DataFrame(texts)

# 1B) INSERT (Blocks: valves/instruments/equipment symbols)
inserts = []
for e in msp.query("INSERT"):
    inserts.append({
        "block": e.dxf.name,
        "x": e.dxf.insert.x,
        "y": e.dxf.insert.y,
        "layer": e.dxf.layer,
        "rotation": float(getattr(e.dxf, "rotation", 0.0)),
        "scale_x": float(getattr(e.dxf, "xscale", 1.0)),
        "scale_y": float(getattr(e.dxf, "yscale", 1.0)),
    })
df_ins = pd.DataFrame(inserts)

# 1C) Lines and polylines (pipes)
pipes = []
for e in msp.query("LINE LWPOLYLINE"):
    if e.dxftype() == "LINE":
        p1 = (e.dxf.start.x, e.dxf.start.y)
        p2 = (e.dxf.end.x, e.dxf.end.y)
        pts = [p1, p2]
    else:
        # LWPOLYLINE points
        pts = [(p[0], p[1]) for p in e.get_points("xy")]
    pipes.append({"type": e.dxftype(), "layer": e.dxf.layer, "pts": pts})

df_pipe = pd.DataFrame(pipes)

print("TEXT:", len(df_text), "INSERT:", len(df_ins), "PIPES:", len(df_pipe))
df_ins.head()


In [ ]:
from shapely.geometry import LineString, Point
import networkx as nx

# Convert pipes to shapely lines
pipe_lines = []
for i,r in df_pipe.iterrows():
    if len(r["pts"]) >= 2:
        pipe_lines.append((i, r["layer"], LineString(r["pts"])))

# Create graph nodes from inserts
G = nx.Graph()
for i,r in df_ins.iterrows():
    G.add_node(
        f"ins_{i}",
        kind="insert",
        block=r["block"],
        layer=r["layer"],
        x=float(r["x"]),
        y=float(r["y"])
    )

# Helper: distance from node to pipe
def node_to_pipe_dist(node_id, pipe_geom):
    x, y = G.nodes[node_id]["x"], G.nodes[node_id]["y"]
    return pipe_geom.distance(Point(x, y))

# Connect nodes that lie close to the same pipe segment
DIST_THRESH = 5.0  # you may tune depending on drawing units
for pid, layer, geom in pipe_lines:
    near = []
    for n in G.nodes:
        d = node_to_pipe_dist(n, geom)
        if d <= DIST_THRESH:
            near.append((n, d))
    near = sorted(near, key=lambda t: t[1])

    # connect near nodes along that pipe (pairwise)
    for a in range(len(near)-1):
        u, du = near[a]
        v, dv = near[a+1]
        if u != v:
            G.add_edge(u, v, edge_type="process_pipe", pipe_id=int(pid), pipe_layer=layer)

print("Graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))
# plot pipes
for _,_,geom in pipe_lines[:500]:  # cap if huge
    xs, ys = geom.xy
    plt.plot(xs, ys, linewidth=0.5)

# plot nodes
xs = [G.nodes[n]["x"] for n in G.nodes]
ys = [G.nodes[n]["y"] for n in G.nodes]
plt.scatter(xs, ys, s=8)

plt.title("DXF P&ID: Pipes + Insert Nodes (sanity check)")
plt.axis("equal")
plt.show()


In [ ]:
xs = df_ins["x"].values
ys = df_ins["y"].values

print("X range:", xs.min(), xs.max())
print("Y range:", ys.min(), ys.max())


In [ ]:
import numpy as np

# Robust center
mx, my = np.median(xs), np.median(ys)

# Distance from center
dist = np.sqrt((xs-mx)**2 + (ys-my)**2)

# Keep only nodes within reasonable radius
KEEP_RADIUS = np.percentile(dist, 95)  # keeps 95% of symbols
mask = dist <= KEEP_RADIUS

df_ins_clean = df_ins[mask].copy()
print("Before:", len(df_ins), "After:", len(df_ins_clean))


In [ ]:
# Normalize to zero mean, unit scale
x_mean, y_mean = df_ins_clean["x"].mean(), df_ins_clean["y"].mean()
x_std,  y_std  = df_ins_clean["x"].std(),  df_ins_clean["y"].std()

df_ins_clean["x_norm"] = (df_ins_clean["x"] - x_mean) / x_std
df_ins_clean["y_norm"] = (df_ins_clean["y"] - y_mean) / y_std


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(df_ins_clean["x_norm"], df_ins_clean["y_norm"], s=10)
plt.title("Normalized P&ID Insert Nodes (Sanity Check)")
plt.axis("equal")
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

coords = df_ins_clean[["x_norm", "y_norm"]].values

# DBSCAN groups spatial clusters
clustering = DBSCAN(eps=0.6, min_samples=5).fit(coords)
df_ins_clean["cluster"] = clustering.labels_

df_ins_clean["cluster"].value_counts()


In [ ]:
main_cluster = df_ins_clean["cluster"].value_counts().idxmax()
df_main = df_ins_clean[df_ins_clean["cluster"] == main_cluster].copy()

print("Keeping cluster:", main_cluster, "Nodes:", len(df_main))


In [ ]:
x_mean = df_main["x"].mean()
y_mean = df_main["y"].mean()
x_std  = df_main["x"].std()
y_std  = df_main["y"].std()

df_main["x_n"] = (df_main["x"] - x_mean) / x_std
df_main["y_n"] = (df_main["y"] - y_mean) / y_std


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(df_main["x_n"], df_main["y_n"], s=15)
plt.title("Final ML-Ready P&ID Geometry (Single Cluster)")
plt.axis("equal")
plt.show()


In [ ]:
import networkx as nx
import numpy as np

G = nx.Graph()

for i,r in df_main.iterrows():
    G.add_node(
        f"ins_{i}",
        block=str(r["block"]),
        layer=str(r["layer"]),
        x=float(r["x_n"]),
        y=float(r["y_n"])
    )

print("Nodes:", G.number_of_nodes())


In [ ]:
from sklearn.neighbors import NearestNeighbors

node_ids = list(G.nodes())
coords = np.array([[G.nodes[n]["x"], G.nodes[n]["y"]] for n in node_ids], dtype=float)

K = 6  # nearest neighbors per node (tune 4–10)
nbrs = NearestNeighbors(n_neighbors=K+1, algorithm="ball_tree").fit(coords)
distances, indices = nbrs.kneighbors(coords)

# add edges excluding self
for i in range(len(node_ids)):
    for jpos in range(1, K+1):
        j = indices[i, jpos]
        d = distances[i, jpos]
        if d < 0.8:  # distance cutoff in normalized coords (tune)
            u, v = node_ids[i], node_ids[j]
            if u != v:
                G.add_edge(u, v, edge_type="knn_candidate", dist=float(d))

print("Edges:", G.number_of_edges())


In [ ]:
import pandas as pd
import json

nodes_df = pd.DataFrame([
    {"id": n, **G.nodes[n]} for n in G.nodes()
])

edges_df = pd.DataFrame([
    {"u": u, "v": v, **attr} for u,v,attr in G.edges(data=True)
])

nodes_df.to_csv("nodes.csv", index=False)
edges_df.to_csv("edges.csv", index=False)

pid_json = {"nodes": nodes_df.to_dict("records"), "edges": edges_df.to_dict("records")}
with open("pid.json","w") as f:
    json.dump(pid_json, f, indent=2)

print("Saved nodes.csv, edges.csv, pid.json")


In [ ]:
deg = np.array([d for _,d in G.degree()])
print("Median degree:", np.median(deg))
print("Max degree:", deg.max())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,7))
for u,v,attr in list(G.edges(data=True))[:800]:
    plt.plot([G.nodes[u]["x"], G.nodes[v]["x"]],
             [G.nodes[u]["y"], G.nodes[v]["y"]],
             linewidth=0.4)

plt.scatter(coords[:,0], coords[:,1], s=10)
plt.title("Graph check: edges are local (KNN)")
plt.axis("equal")
plt.show()
